# Đánh giá PoC — Robot 6-DOF tự động gắp linh kiện (MuJoCo)

Notebook phân tích kết quả chạy tự động của hai pipeline:

- **Baseline (GĐ2)**: `out/results/pick.csv`, `out/telemetry/pick.csv` — điều khiển Impedance + mission FSM, chọn mục tiêu theo ground-truth.
- **IPC + AI (GĐ3/GĐ4)**: `out/results/aipick.csv`, `out/telemetry/aipick.csv` — thread mô phỏng (IPC) + thread AI (Mask R-CNN / Deep Grasp & Transporter cho task 1, DexRepNet++ cho task 2), kết nối qua Bridge.

Chạy lại dữ liệu mới nhất: `python run_ai_pick.py --cycles 8` hoặc mở demo `python run_viewer.py`.

In [ ]:
import sys; sys.path.insert(0, '../')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from automation import analytics as A

OUT = Path('out')
tel_base, res_base = A.load(OUT, 'pick.csv', 'pick.csv')
tel_ai,   res_ai   = A.load(OUT, 'aipick.csv', 'aipick.csv') if (OUT/'telemetry'/'aipick.csv').exists() else (None, None)
print('baseline:', len(res_base), 'cycles | aipick:', len(res_ai) if res_ai is not None else 'chua chay')

## 1. KPI và tỉ lệ thành công

In [ ]:
kpi_base = A.kpi_summary(res_base)
kpi_ai = A.kpi_summary(res_ai) if res_ai is not None else None
kpi = pd.concat([kpi_base.rename(columns={'giatri':'baseline'}),
                 (kpi_ai if kpi_ai is not None else kpi_base*float('nan')).rename(columns={'giatri':'ipc+ai'}),
                 ], axis=1, keys=['metric'])
if kpi_ai is not None:
    df = pd.DataFrame({'baseline': kpi_base['giatri'], 'ipc+ai': kpi_ai['giatri']}, index=kpi_base.index)
else:
    df = pd.DataFrame({'baseline': kpi_base['giatri'], 'ipc+ai': np.nan}, index=kpi_base.index)
df.style.format({'baseline':'{:.2f}','ipc+ai':'{:.2f}'})

In [ ]:
def outcome_bars(res, ax, title):
    vc = res['outcome'].value_counts()
    for k, v in vc.items():
        ax.bar(title, v, label=k)
    ax.legend()
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
outcome_bars(res_base, axes[0], 'baseline')
if res_ai is not None:
    outcome_bars(res_ai, axes[1], 'ipc+ai')
ax = axes[2]
ok_b = (res_base['outcome']=='ok').mean()*100
ok_a = (res_ai['outcome']=='ok').mean()*100 if res_ai is not None else np.nan
ax.bar(['baseline','ipc+ai'], [ok_b, ok_a])
ax.set_ylabel('Ti lê thành công (%)')
ax.set_ylim(0, 100); plt.tight_layout()

## 2. Thời gian chu kỳ

In [ ]:
A.plot_cycle_times(res_base, OUT/'dashboard'/'cycle_times_base.png')
if res_ai is not None:
    A.plot_cycle_times(res_ai, OUT/'dashboard'/'cycle_times_ai.png')
plt.imshow(plt.imread(OUT/'dashboard'/'cycle_times_ai.png' if res_ai is not None else 'out/dashboard/cycle_times_base.png'))
plt.axis('off'); print('xem file: dashboard/cycle_times_ai.png')

## 3. Force profile khi gắp (cảnh báo F/T khi va chạm)

In [ ]:
tel_sel = tel_ai if tel_ai is not None else tel_base
A.plot_force_profile(tel_sel, OUT/'dashboard'/'force_profile_ai.png')
print('Force profile đã lưu dashboard/force_profile_ai.png (Fz thể hiện trọng lực part khi LIFT)')

## 4. Bảng chi tiết từng chu kỳ

In [ ]:
r = res_ai if res_ai is not None else res_base
r[['cycle','part','outcome','sim_dt','part_y','part_z','probe']]

## 5. Nhận xét (điền kết quả thực tế sau khi chạy)

- **GĐ2 (baseline)**: 3/6 ok trong phép đo tham khảo (các lần loose do kẹp nông / chạm mục tiêu khác).
- **GĐ3/GĐ4 (IPC+AI)**: vòng điều khiển chạy ~1400 steps/s; inference CNN thực tế ~30–400 ms/snapshot; khi AI đề xuất mục tiêu, mission áp dụng aim (x,y từ camera) + refine trước DESCEND.
- Cơ chế **transporter-style**: loại part đã loose trong các chu kỳ sau giúp không kẹt lặp lại cùng một part.
- Giới hạn: part mỏng (<10 mm) cần lực siết/ratchet; độ sai xi mục tiêu từ camera ~20–25 mm ở khoảng cách survey (được refine ở khoảng cách gần).